# (노트) gym

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [강화학습]

In [3]:
import gym
from gym import spaces
import numpy as np
from typing import Optional
#from gym.utils.renderer import Renderer
import pygame 

pygame 2.1.0 (SDL 2.0.16, Python 3.7.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array", "single_rgb_array"], "render_fps": 4}

    def __init__(self, render_mode: Optional[str] = None, size: int = 5):
        assert render_mode is None or render_mode in self.metadata["render_modes"]
        self.render_mode = render_mode  # Define the attribute render_mode in your environment

        self.size = size  # The size of the square grid
        self.window_size = 512  # The size of the PyGame window

        # Observations are dictionaries with the agent's and the target's location.
        # Each location is encoded as an element of {0, ..., `size`}^2, i.e. MultiDiscrete([size, size]).
        self.observation_space = spaces.Dict(
            {
                "agent": spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": spaces.Box(0, size - 1, shape=(2,), dtype=int),
            }
        )

        # We have 4 actions, corresponding to "right", "up", "left", "down"
        self.action_space = spaces.Discrete(4)

        """
        The following dictionary maps abstract actions from `self.action_space` to 
        the direction we will walk in if that action is taken.
        I.e. 0 corresponds to "right", 1 to "up" etc.
        """
        self._action_to_direction = {
            0: np.array([1, 0]),
            1: np.array([0, 1]),
            2: np.array([-1, 0]),
            3: np.array([0, -1]),
        }

        """
        If human-rendering is used, `self.window` will be a reference
        to the window that we draw to. `self.clock` will be a clock that is used
        to ensure that the environment is rendered at the correct framerate in
        human-mode.
        """
        if self.render_mode == "human":
            import pygame  # import here to avoid pygame dependency with no render

            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
            self.clock = pygame.time.Clock()
                
        # The following line uses the util class Renderer to gather a collection of frames 
        # using a method that computes a single frame. We will define _render_frame below.
        self.renderer = Renderer(self.render_mode, self._render_frame)
    def _get_obs(self):
        return {"agent": self._agent_location, "target": self._target_location}
    def _get_info(self):
        return {"distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)}
    def reset(self, seed=None, return_info=False, options=None):
        # We need the following line to seed self.np_random
        super().reset(seed=seed)

        # Choose the agent's location uniformly at random
        self._agent_location = self.np_random.integers(0, self.size, size=2)

        # We will sample the target's location randomly until it does not coincide with the agent's location
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(0, self.size, size=2)

        # clean the render collection and add the initial frame
        self.renderer.reset()
        self.renderer.render_step()

        observation = self._get_obs()
        info = self._get_info()
        return (observation, info) if return_info else observation
    def step(self, action):
        # Map the action (element of {0,1,2,3}) to the direction we walk in
        direction = self._action_to_direction[action]
        # We use `np.clip` to make sure we don't leave the grid
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )
        # An episode is done if the agent has reached the target
        done = np.array_equal(self._agent_location, self._target_location)
        reward = 1 if done else 0  # Binary sparse rewards
        observation = self._get_obs()
        info = self._get_info()

        # add a frame to the render collection
        self.renderer.render_step()

        return observation, reward, done, info
    def render(self):
        # Just return the list of render frames collected by the Renderer.
        return self.renderer.get_renders()

    def _render_frame(self, mode: str):
        # This will be the function called by the Renderer to collect a single frame.
        assert mode is not None  # The renderer will not call this function with no-rendering.
    
        import pygame # avoid global pygame dependency. This method is not called with no-render.
    
        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_square_size = (
            self.window_size / self.size
        )  # The size of a single grid square in pixels

        # First we draw the target
        pygame.draw.rect(
            canvas,
            (255, 0, 0),
            pygame.Rect(
                pix_square_size * self._target_location,
                (pix_square_size, pix_square_size),
            ),
        )
        # Now we draw the agent
        pygame.draw.circle(
            canvas,
            (0, 0, 255),
            (self._agent_location + 0.5) * pix_square_size,
            pix_square_size / 3,
        )

        # Finally, add some gridlines
        for x in range(self.size + 1):
            pygame.draw.line(
                canvas,
                0,
                (0, pix_square_size * x),
                (self.window_size, pix_square_size * x),
                width=3,
            )
            pygame.draw.line(
                canvas,
                0,
                (pix_square_size * x, 0),
                (pix_square_size * x, self.window_size),
                width=3,
            )

        if mode == "human":
            assert self.window is not None
            # The following line copies our drawings from `canvas` to the visible window
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()

            # We need to ensure that human-rendering occurs at the predefined framerate.
            # The following line will automatically add a delay to keep the framerate stable.
            self.clock.tick(self.metadata["render_fps"])
        else:  # rgb_array or single_rgb_array
            return np.transpose(
                np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2)
            )
    def close(self):
        if self.window is not None:
            import pygame 
            
            pygame.display.quit()
            pygame.quit()

In [5]:
env=GridWorldEnv()

NameError: name 'Renderer' is not defined

In [28]:
#import gym
#env = gym.make("LunarLander-v2", render_mode="human")
env.action_space.seed(42)

observation, info = env.reset(seed=42, return_info=True)

for _ in range(200): # 200 is time 
    observation, reward, done, info = env.step(env.action_space.sample())

    if done:
        observation, info = env.reset(return_info=True)

env.close()

AttributeError: 'GridWorldEnv' object has no attribute 'window'

In [29]:
import gym
import pygame
from gym.utils.play import play
mapping = {'a':1, 'w':2, 'd':3, 's':0}
env = GridWorldEnv()
play(env, keys_to_action=mapping)

/home/cgb4/anaconda3/envs/py37/lib/python3.7/site-packages/gym/utils/play.py:215: DeprecationWarning: WARN: `play.py` currently supports only the old step API which returns one boolean, however this will soon be updated to support only the new step api that returns two bools.
  "`play.py` currently supports only the old step API which returns one boolean, however this will soon be updated to support only the new step api that returns two bools."
/home/cgb4/anaconda3/envs/py37/lib/python3.7/site-packages/gym/core.py:52: DeprecationWarning: WARN: The argument mode in render method is deprecated; use render_mode during environment initialization instead.
See here for more information: https://www.gymlibrary.ml/content/api/
  "The argument mode in render method is deprecated; "


TypeError: render() got an unexpected keyword argument 'mode'

In [30]:
import gym
import pygame
from gym.utils.play import play
mapping = {'a':1, 'w':2, 'd':3, 's':0}
env=gym.make("LunarLander-v2")
#play(env, keys_to_action=mapping)
play(env, keys_to_action=mapping)

/home/cgb4/anaconda3/envs/py37/lib/python3.7/site-packages/gym/core.py:330: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  "Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future."
/home/cgb4/anaconda3/envs/py37/lib/python3.7/site-packages/gym/wrappers/step_api_compatibility.py:40: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  "Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future."


In [10]:
env.env.render(mode="rgb_array")

AttributeError: 'GridWorldEnv' object has no attribute 'env'

    0- Do nothing
    1- Fire left engine
    2- Fire down engine
    3- Fire right engine

In [11]:
env.action_space

Discrete(4)

- 4 actions

`-` PLAY GAME

In [6]:
import numpy as np

In [7]:
import gym
from gym.utils.play import play
env =gym.make("CarRacing-v2") 
play(env, 
     keys_to_action={"w": np.array([0, 0.7, 0]),
                     "a": np.array([-1, 0, 0]),
                     "s": np.array([0, 0, 1]),
                     "d": np.array([1, 0, 0]),
                     "wa": np.array([-1, 0.7, 0]),
                     "dw": np.array([1, 0.7, 0]),
                     "ds": np.array([1, 0, 1]),
                     "as": np.array([-1, 0, 1]),}, noop=np.array([0,0,0]))

DeprecatedEnv: Env CarRacing-v2 not found (valid versions include ['CarRacing-v0'])